# Signals and systems: intuition before application
**Python, vectors, matrices, dot products, and basic calculus assumed**

**Part I, Labs 1–4, studies signals on their own.** We begin with a simple rule connecting time and value, then build up to waveforms, sequences, systems, and frequency. All Part I examples use dimensionless mathematical values. First sketch or predict what should happen, then use a small implementation and a supplied plot to test the idea. The project dataset and application pipeline appear only in Part II, after the fundamentals checkpoint.

| Topic | Reading and practice |
|---|---|
| [Introduction](#foundations) | Read a graph, evaluate a rule, distinguish time from index |
| [1. Waveforms](#lab-1) | Amplitude, frequency, phase, offset, and delay |
| [1A. Timing and size](#bridge-1) | Elementary sequences, statistics, and boundary assumptions |
| [2. Sampling](#lab-2) | Samples, aliasing, timestamps, and interpolation |
| [2A. Acquisition](#bridge-2) | Quantization, clipping, and information loss |
| [3. Systems](#lab-3) | Filtering, causality, and convolution |
| [3A. Convolution in detail](#bridge-3) | Individual contributions, boundaries, and noise |
| [4. Frequency](#lab-4) | Fourier analysis and filter response |
| [4A–4B. Spectra and reconstruction](#bridge-4) | Complex geometry, amplitude scaling, and inverse DFT |
| [Fundamentals checkpoint](#part-ii) | Explain the concepts before applying them |
| [5. Multichannel processing](#lab-5) | Apply the fundamentals to a concrete measurement |
| [6. Pipeline evaluation](#lab-6) | Test decisions and explain limitations |

**Fundamentals outcomes:** describe amplitude, offset, frequency, and phase; distinguish sampled values from the underlying model; explain aliasing; predict a system's effect on pulses and sinusoids; connect convolution with frequency response.

Each lab follows **predict → implement → run → explain**. Functions marked **TODO** are intentionally unfinished. Supplied checks and plotting cells catch `NotImplementedError` and explain what to complete; they never substitute answers. Incorrect implementations fail their checks. The opening challenge evaluates a simple signal. Lab 1 then builds waveforms and computes statistics. Projection and cosine similarity appear in the final optional extension. Labs 4–6 reuse the FIR function from Lab 3.

Use `signals_workshop_solutions.ipynb` to compare after trying. All examples work offline after dependency installation. Each reading names its companion lecture slides so you can alternate between the two.


Work through the topics in order, alternating the lecture with its reading, predictions, coding challenge, and discussion. The concept bridges are part of the continuous progression. Return to a diagram or explanation whenever a checkpoint reveals a gap.


## Setup for Part I
Install once from this folder: `python -m pip install -r requirements.txt`, then launch `python -m jupyter lab signals_basics_coding_challenges.ipynb`. Use a Python 3.10+ kernel.

The next cell imports NumPy and Matplotlib and defines a small check helper. Each fundamentals lab creates its own deterministic example. All examples in Part I are defined by the notebook itself. Parameters in a lab affect that lab's example; subsequent labs construct their own inputs.

Throughout Part I, vertical axes show dimensionless signal values. We introduce a new axis or notation when we need it, beginning with time in seconds and an ordinary graph.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11,
                     "axes.grid": True, "grid.alpha": 0.2,
                     "axes.spines.top": False, "axes.spines.right": False})

def need(ready, message):
    if not ready:
        raise NotImplementedError(message)


<a id="foundations"></a>
## Part I · What is a signal?
**Lecture companion: “What is a signal?” (slide 2) → “Reading one point” (slide 3)**

Start with a value that can vary over time. At each time we can ask: **what is the value now?** The complete relationship between time and value is a **signal**.

For our first example, the value starts at 1 and increases by 0.5 each second. After 2 seconds the value is 2. These are just mathematical values; no measurement device or physical units are needed.

On a graph, find a time on the horizontal axis, move vertically to the curve, then read the value on the vertical axis. The point (2, 2) means **time 2 seconds, value 2**.

**Predict:** what value should appear at time 4 seconds? Is a point's horizontal coordinate the same thing as its vertical coordinate?


### Give the relationship a name
**Lecture companion: “The notation x(t)” (slide 4) → “A rule, a table, and a graph” (slide 5)**

We call this signal $x$. The notation $x(t)$ means **the value of signal x at time t**. Parentheses here mean “evaluate the function,” not multiplication. For this example:

$$x(t)=1+\frac{t}{2},\qquad 0\leq t\leq4.$$

The rule, a table of evaluations, and a graph are three ways to describe the same relationship.

| Time t (s) | 0 | 1 | 2 | 3 | 4 |
|---|---:|---:|---:|---:|---:|
| Signal value x(t) | 1 | 1.5 | 2 | 2.5 | 3 |

A formula also tells us about times between these entries: for example, $x(1.5)=1.75$. A table by itself does not specify a rule between its listed entries.


### Many shapes count as signals
**Lecture companion: “Signals can have many shapes” (slide 6) → “Continuous time can include a jump” (slide 8)**

A signal can stay constant, become negative, rise gradually, repeat, or jump abruptly. “Signal” does not mean “sine wave.” A negative value is a valid value; zero is another valid value.

A **continuous-time signal** is defined at every real-valued time in its domain, including times such as 1.5 or 1.501. The word “continuous” describes the available times. The values can still jump. For example, a rule that is 0 before time 2 and 1 from time 2 onward is defined at every time, even though its graph jumps.

**Sketch:** draw a constant signal, one with negative values, and one with a jump. State the value at the jump itself. A short displayed interval is only the part of the signal we chose to show.


### Discrete time: an ordered sequence
**Lecture companion: “Discrete time” (slide 9) → “Lines between samples add an assumption” (slide 11)**

A **discrete-time signal** is defined at integer positions. We write $x[n]$, using square brackets as a reminder that n is an integer index. For the sequence `[1., 1.5, 2., 2.5, 3.]`, $x[2]=2$. Index 2 is a position in a list; it does not by itself mean 2 seconds.

A stem plot places a dot at each defined index and a vertical line beneath it to make the value easy to see. There is no value at index 2.5 unless we introduce a separate rule. Connecting the dots is a drawing choice; it does not add values to the sequence.

**Explain:** what extra information would you need to put times in seconds beside the indices? We will build that connection in Lab 2. For now, distinguish a real-valued time from an integer position.


### Opening coding challenge · Evaluate a signal
**Lecture companion: “Opening challenge · Evaluate a signal” (slide 12)**

Implement `evaluate_signal(t)` for the rule **1 + t/2**. Accept either a number or a NumPy array and evaluate the rule element by element. You may assume the requested times lie between 0 and 4 seconds.

**Predict → implement → run → explain:** calculate the values at 0, 2, and 4 by hand, complete the TODO, and run the supplied plotting cell. The left graph evaluates a continuous-time rule on a fine plotting grid. The right graph displays a finite sequence by index. A computer draws using finitely many points; the mathematical rule defines values between them.


In [ ]:
def evaluate_signal(t):
    # TODO: evaluate the rule 1 + t/2 for one time or a NumPy array of times.
    raise NotImplementedError("evaluate_signal")


In [3]:
try:
    np.testing.assert_allclose(evaluate_signal(2.), 2.)
    np.testing.assert_allclose(evaluate_signal(np.array([0., 2., 4.])), [1., 2., 3.])
    # EDIT: probe_time. Predict its value before rerunning.
    probe_time = 2.
    assert 0 <= probe_time <= 4, "Choose a time within the rule's domain."
    plot_time = np.linspace(0, 4, 401)
    sequence_values = evaluate_signal(np.arange(5.))
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), layout='constrained')
    axes[0].plot(plot_time, evaluate_signal(plot_time))
    probe_value = evaluate_signal(probe_time)
    axes[0].scatter([probe_time], [probe_value], color='tab:orange', zorder=3)
    axes[0].plot([probe_time, probe_time], [0, probe_value], '--', color='tab:orange', alpha=.7)
    axes[0].annotate(f't = {probe_time:g} s, value = {probe_value:g}',
                     (probe_time, probe_value), xytext=(12, 16), textcoords='offset points')
    axes[0].set(xlabel='Time t (s)', ylabel='Signal value', title='A rule at any time in its domain', ylim=(0, 3.8))
    axes[1].stem(np.arange(5), sequence_values, basefmt=' ')
    axes[1].set(xlabel='Index n', ylabel='Signal value', title='A finite sequence at integer positions',
                xticks=np.arange(5), ylim=(0, 3.8))
    plt.show()
except NotImplementedError as error:
    print('Opening challenge is waiting for your implementation:', error)


Opening challenge is waiting for your implementation: evaluate_signal


<details><summary>Worked explanation · compare after predicting</summary>

The rule gives values 1, 2, and 3 at times 0, 2, and 4 seconds. Changing the probe changes which point we inspect, not the underlying rule. The left graph has time on its horizontal axis; the right has integer positions. The five sequence values happen to be evaluations of the rule at five selected times, but an array alone does not carry those times.

</details>


<a id="lab-1"></a>
## 1. Waveforms: change one property at a time
**Lecture companion: “A repeating pattern” (slide 16) → “Period and frequency” (slide 17)**

A **waveform** is the shape of a signal's graph. A **periodic** signal repeats: shifting it by one period leaves the values unchanged. Its **period** T is the time for one repeat, measured in seconds. Its **frequency** f is the number of repeats per second, measured in hertz (Hz): $f=1/T$.

A waveform repeating every 0.5 seconds has frequency 2 Hz. Doubling its frequency squeezes twice as many cycles into the same interval. A finite graph may look repetitive without proving that the complete signal repeats forever.

**Predict:** if the period changes from 0.5 seconds to 0.25 seconds, what happens to the frequency?


### Height and centerline are different properties
**Lecture companion: “Amplitude changes the height” (slide 18) → “Offset moves the centerline” (slide 19)**

For a sinusoid, **amplitude** A is the distance from its centerline to a peak. We take A ≥ 0. A wave with amplitude 2 extends 2 above and 2 below its centerline; its peak-to-peak size is 4.

The **offset** b moves the centerline. With b = 3 and A = 2, the peaks are at 5 and the troughs at 1. Changing offset does not change the spacing of peaks.

**Predict:** compare doubling amplitude with adding 2 to every value. Which changes the peak-to-peak size? Which moves the centerline?


### Where are we within a cycle?
**Lecture companion: “Angles describe progress through a cycle” (slide 20) → “Phase selects a point within the cycle” (slide 21)**

One complete turn is $2\pi$ **radians**. Half a turn is $\pi$ and a quarter turn is $\pi/2$. A sine function turns progress around that cycle into a value between −1 and 1. At angles 0, $\pi/2$, $\pi$, and $3\pi/2$, its values are 0, 1, 0, and −1.

At frequency f, the angle advances by $2\pi f$ radians each second. The **phase** $\phi$ selects the starting position in that cycle. Adding $\pi/2$ starts the sine at its positive peak instead of its rising zero crossing. It does not change how long a cycle takes.

**Predict:** can two waves have the same amplitude and frequency while reaching their peaks at different times?


### Put the waveform parameters together
**Lecture companion: “The parameters of a sinusoid” (slide 22) → “A delay shifts when values occur” (slide 23)**

The waveform $b+A\sin(2\pi ft+\phi)$ combines offset, amplitude, frequency, and phase. Read it inside out: form an angle from time, evaluate sine, scale the result, then add the offset.

A **delay** shifts when values occur. Replacing t with $t-\tau$ moves the waveform right when $\tau>0$. A feature formerly at t = 0.5 seconds occurs at t = 0.7 seconds after a 0.2-second delay. Our complete function is:

$$x(t)=b+A\sin\bigl(2\pi f(t-\tau)+\phi\bigr).$$

For a single sinusoid, delaying by $\tau$ also changes its phase by $-2\pi f\tau$. This relation depends on frequency. Shifting every value in a complicated waveform later is a time delay, even when it has no single phase parameter.

**Predict:** sketch the effect of changing A, f, b, $\phi$, and $\tau$ separately. Keep all other parameters fixed.


### Combine and summarize waveforms
**Lecture companion: “Complicated shapes can have simple components” (slide 26) → “Mean and RMS answer different questions” (slide 30)**

Signals add at the same time: $(x+z)(t)=x(t)+z(t)$. A slow sine plus a fast sine makes a more complicated shape. Each component retains its own frequency; addition does not add their frequencies.

A finite sequence is a vector of values. Put several sequences in columns of an N×C array: N positions and C example signals. Their **mean** measures average level. Their **root mean square (RMS)** measures size:

$$\mathrm{RMS}(x)=\sqrt{\frac1N\sum_{n=0}^{N-1}x[n]^2}.$$

Squaring removes sign, averaging summarizes the record, and taking the square root restores the original scale. A zero-centered oscillation can have mean zero and nonzero RMS. Over complete cycles, a zero-offset sine of amplitude A has RMS $A/\sqrt2$.

### Coding challenge 1
- `sine_wave(t, amplitude, frequency, phase=0., offset=0., delay=0.)`: implement the formula for a NumPy time array; preserve its shape. Phase uses radians, frequency uses Hz, and delay uses seconds.
- `channel_stats(X)`: return `(mean, rms)` over rows, each of length C. Here “channel” simply means one column containing an example signal.

The next experiment supplies the plots. Change one parameter at a time, explain what moved, then compare the mean and RMS of x, −x, and x+2.


In [4]:
def sine_wave(t, amplitude, frequency, phase=0., offset=0., delay=0.):
    # TODO: construct a sinusoid from its amplitude, frequency, phase, offset, and delay.
    raise NotImplementedError("sine_wave")

def channel_stats(X):
    # TODO: summarize each signal column over its rows.
    raise NotImplementedError("channel_stats")


In [5]:
LAB_1_READY = False
try:
    # Check meaning: zero phase, quarter cycle, delay, and DC level.
    times = np.array([0., .125, .25])
    np.testing.assert_allclose(sine_wave(times, 3, 2), [0, 3, 0], atol=1e-12)
    np.testing.assert_allclose(sine_wave(times, 3, 2, phase=np.pi/2), [3, 0, -3], atol=1e-12)
    np.testing.assert_allclose(sine_wave(times, 3, 2, delay=.125), [-3, 0, 3], atol=1e-12)
    np.testing.assert_allclose(sine_wave(times, 0, 2, offset=2), [2, 2, 2])
    np.testing.assert_equal(sine_wave(times, 3, 2).shape, times.shape)

    # EDIT THESE PARAMETERS. Change one at a time after predicting its effect.
    amplitude_1, frequency_1, phase_1, offset_1, delay_1 = 2., 2., 0., 0., 0.
    t1 = np.arange(400) / 200  # fine plotting grid, not a true continuum
    reference1 = sine_wave(t1, 2., 2.)
    edited1 = sine_wave(t1, amplitude_1, frequency_1, phase_1, offset_1, delay_1)
    fast1 = sine_wave(t1, .6, 9.)
    X1 = np.column_stack([reference1, -reference1, reference1 + 2])
    mean1, rms1 = channel_stats(X1)
    np.testing.assert_allclose(mean1, [0, 0, 2], atol=1e-12)
    np.testing.assert_allclose(rms1, [np.sqrt(2), np.sqrt(2), np.sqrt(6)], atol=1e-12)
    fig, axes = plt.subplots(1, 2, figsize=(12,4), layout='constrained')
    axes[0].plot(t1, reference1, '--', label='Reference: A=2, f=2 Hz')
    axes[0].plot(t1, edited1, label='Your parameter choices')
    axes[1].plot(t1, reference1, alpha=.6, label='2 Hz component')
    axes[1].plot(t1, fast1, alpha=.6, label='9 Hz component')
    axes[1].plot(t1, reference1 + fast1, label='Pointwise sum')
    for ax in axes:
        ax.set(xlabel='Time (s)', ylabel='Signal value'); ax.legend(fontsize=9)
    axes[0].set_title('One change at a time'); axes[1].set_title('Components make a waveform')
    plt.show()
    print('Means of [x, -x, x+2]:', np.round(mean1, 3))
    print('RMS of [x, -x, x+2]:', np.round(rms1, 3))
    LAB_1_READY = True
except NotImplementedError as error:
    print('Lab 1 core is waiting for your implementation:', error)


Lab 1 core is waiting for your implementation: sine_wave


### Investigate and record
1. Change each sinusoid parameter separately. For each, state what moved on the graph and what stayed fixed. Explain why a positive delay shifts the curve to the right.
2. Compare `x`, `-x`, and `x+2` using mean and RMS. Why can the average of an oscillation be zero even though it is not a zero signal?
3. In the sum plot, identify the slow undulation and fast ripple. Predict what a smoother might remove and what it might accidentally change.

**Deliverable:** waveform and statistics functions, annotated predictions, and a short explanation of frequency versus phase.


<a id="bridge-1"></a>
### 1A. Timing, elementary signals, and size
**Lecture companion: “Time scaling and reversal change the horizontal axis” (slide 24) → “Discrete delays require a boundary convention” (slide 25) → “Variation around the mean differs from total size” (slide 32)**

The **domain** says where a signal is defined (real time, integer indices, or pixel coordinates); the **values** say what quantity it takes there. A continuous-time signal can jump. A discrete-time signal can still have real-valued amplitudes. A finite record only observes a window of an underlying signal; apparent repetition in that window does not prove repetition forever.

A period T seconds corresponds to frequency $f=1/T$ Hz and angular frequency $\Omega=2\pi f$ rad/s.

For a continuous-time model, $x(2t)$ compresses durations by two, $x(t-\tau)$ delays by $\tau>0$, and $x(-t)$ reverses time. Follow a distinctive feature: if x peaks at t=0.6 s, then x(2t) peaks at 0.3 s. A discrete delay $x[n-d]$ needs a choice for samples outside the record. Zero prehistory and periodic wraparound are different assumptions.

Useful elementary sequences are the unit step $u[n]$ (one for n≥0), the unit impulse $\delta[n]$ (one only at zero), and the decaying exponential $a^nu[n]$ for 0<a<1. A rectangular pulse is a difference of steps; $u[n-2]-u[n-5]$ is nonzero at indices 2, 3, 4. The discrete impulse here is not the continuous-time Dirac delta.

For a real finite record, mean measures level, population standard deviation measures variation about that level, and RMS measures total size. They satisfy $\mathrm{RMS}^2=\mu^2+\sigma^2$. Finite-record energy is $\sum x[n]^2$ and average power is that sum divided by N. These are mathematical measures of signal size. Here the values are dimensionless; a connection to physical energy would require a separate model.

**Predict:** delay `[1, 3, 2, 0, 0, 4]` by two samples, assuming zero prehistory and keeping the same output length. Which values disappear from this finite output? How would a circular shift differ?

### Coding challenge 1A
Implement `delay_zero(x, d)` for a real 1D array and a nonnegative integer delay. Return the same length, with zeros for unavailable history. A delay at least as long as x returns all zeros. Do not use `np.roll`. The supplied plot compares your result with a circular shift.


In [6]:
def delay_zero(x, d):
    # TODO: delay by d samples with zero prehistory, preserving the original length.
    raise NotImplementedError("delay_zero")


In [7]:
try:
    source_a = np.array([1., 3, 2, 0, 0, 4])
    np.testing.assert_allclose(delay_zero(source_a, 2), [0, 0, 1, 3, 2, 0])
    np.testing.assert_allclose(delay_zero(source_a, 0), source_a)
    np.testing.assert_allclose(delay_zero(source_a, 9), np.zeros(6))
    # EDIT: delay_a; this example never changes inputs in another lesson.
    delay_a = 2
    fig, axs = plt.subplots(1, 3, figsize=(12,3), layout='constrained')
    for ax, values, title in zip(axs, [source_a, delay_zero(source_a, delay_a), np.roll(source_a, delay_a)], ['Original', 'Zero prehistory', 'Circular shift']):
        ax.stem(np.arange(len(values)), values, basefmt=' ')
        ax.set(xlabel='Sample index n', ylabel='Signal value', title=title, ylim=(-.3, 4.8))
    plt.show()
    print('RMS² = mean² + variance:', np.mean(source_a**2), source_a.mean()**2 + source_a.var())
except NotImplementedError as error:
    print('Challenge 1A is waiting:', error)


Challenge 1A is waiting: delay_zero


<details><summary>Worked explanation · compare after predicting</summary>

The delay gives `[0,0,1,3,2,0]`; the last two original values lie beyond the retained output interval. A circular shift instead places them at the beginning. This is a boundary assumption, not an interchangeable implementation detail. A positive delay moves events later. Offset changes mean and RMS but not population standard deviation.

</details>

<a id="lab-2"></a>
## 2. Sampling: choose times, then evaluate
**Lecture companion: “A function becomes a sequence” (slide 37) → “Sample rate and recording length answer different questions” (slide 41)**

In the opening, a rule gave a value at any time. **Sampling** keeps only values at selected times. For a fixed interval $T_s$, the selected times are $0,T_s,2T_s,\ldots$. The sequence is $x[n]=f(nT_s)$: n identifies a position; $nT_s$ supplies its time.

An interval of 0.05 seconds means **20 samples per second**. The **sample rate** is $f_s=1/T_s$, measured in samples/s or Hz. This is a rate of observations. A waveform's frequency counts its own cycles. The two rates describe different things.

For this lab, the supplied rule is $f(t)=2+3\sin(2\pi\cdot2t)$. It is a 2 Hz sinusoid. We will observe it every 0.05 seconds for a record beginning at 0 and ending just before 2 seconds.

### Coding challenge 2 · Build the sampled array
Implement `sample_function(f, dt, duration)` returning `(t, x)` for `0 <= t < duration` at spacing dt. Assume positive dt and duration. Build the time array, then evaluate f at those times. Use the supplied plot without writing plotting code.

**Predict:** how many values will be in this record? What is the last timestamp? Sketch the curve and your expected dots before running.


In [8]:
def sample_function(f, dt, duration):
    # TODO: create timestamps and evaluate the function at each instant.
    raise NotImplementedError("sample_function")


In [9]:
LAB_2_SAMPLING_READY = False
try:
    f = lambda t: 2 + 3*np.sin(2*np.pi*2*t)
    check_t, check_x = sample_function(f, .05, 2.)
    np.testing.assert_allclose(check_t, np.arange(40)*.05)
    np.testing.assert_allclose(check_x, f(check_t))
    # EDIT: sample_interval. First try .05, then .5; predict before each run.
    sample_interval = .05
    sample_t, sample_x = sample_function(f, sample_interval, 2.)
    dense_t = np.linspace(0, 2, 2000)
    fig, ax = plt.subplots(layout='constrained')
    ax.plot(dense_t, f(dense_t), label='Continuous-time rule')
    ax.scatter(sample_t, sample_x, s=24, label=f'Samples every {sample_interval:g} s', zorder=3)
    ax.set(xlabel='Time (s)', ylabel='Signal value', title='You built the sampled array')
    ax.legend(); plt.show()
    print('Number of samples:', len(sample_t), '| last timestamp:', sample_t[-1], 's')
    LAB_2_SAMPLING_READY = True
except NotImplementedError as error:
    print('Sampling challenge is waiting for your implementation:', error)


Sampling challenge is waiting for your implementation: sample_function


### Relate cycles to observations
**Lecture companion: “Cycles per second and radians per sample” (slide 39)**

A wave with frequency f advances by $2\pi f$ radians each second. Each observation interval lasts $1/f_s$ seconds, so the angle advances by $\omega=2\pi f/f_s$ **radians per sample**. A 2 Hz wave observed at 20 samples/s advances by $0.2\pi$ radians per sample: ten sample intervals span a cycle.

Keep three quantities separate: f counts waveform cycles per second; $f_s$ counts samples per second; $\omega$ describes the waveform's angle change per sample. Increasing the record duration keeps all three unchanged and observes more cycles.

**Check:** the 40 samples in the default experiment occupy indices 0 through 39. Their last timestamp is 1.95 seconds. The requested interval has length 2 seconds, but the first-to-last timestamp span is 1.95 seconds.


### Different curves can give the same samples
**Lecture companion: “Different waves can agree at every sample” (slide 42) → “An alias is an ambiguity in the data” (slide 43)**

At 0.5-second intervals, a 2 Hz sine completes one full cycle between observations. Every sampled value is therefore the same. The curve changes between those observations, but the sequence hides that change.

This ambiguity is **aliasing**. Sine repeats after any whole-number multiple of $2\pi$: $\sin(\theta+2\pi m)=\sin(\theta)$. At sample times $t=n/f_s$, adding $f_s$ to the signal frequency adds $2\pi n$ to the angle. That is a whole number of cycles, so all samples match.

For example, 35 Hz sampled at 50 Hz agrees with a **signed −15 Hz sine** at the sample times. A negative frequency here is a convenient algebraic label: $\sin(-\theta)=-\sin(\theta)$. Replacing it by positive 15 Hz without changing sign or phase would give the wrong sine.

### Coding challenge 2 · Find the alias
Implement `alias_hz(f, fs)` by folding a frequency into the signed interval `[-fs/2, fs/2)`, for positive fs. Test 35 Hz at a 50 Hz sample rate. The provided graph compares the two continuous curves and their shared samples.

**Predict:** do the curves need to agree between the sample times? Can an operation on those shared samples identify which original curve generated them?


In [10]:
def alias_hz(f, fs):
    # TODO: fold frequency into the signed Nyquist interval.
    raise NotImplementedError("alias_hz")


In [11]:
try:
    assert alias_hz(35, 50) == -15
    # EDIT: source_frequency. Keep this example independent of the earlier sampling cell.
    source_frequency, alias_sample_rate = 35., 50.
    folded_frequency = alias_hz(source_frequency, alias_sample_rate)
    alias_t = np.arange(11)/alias_sample_rate
    source_values = np.sin(2*np.pi*source_frequency*alias_t)
    folded_values = np.sin(2*np.pi*folded_frequency*alias_t)
    np.testing.assert_allclose(source_values, folded_values, atol=1e-12)
    dense_alias_t = np.linspace(0, alias_t[-1], 2000)
    fig, ax = plt.subplots(layout='constrained')
    ax.plot(dense_alias_t, np.sin(2*np.pi*source_frequency*dense_alias_t), label=f'{source_frequency:g} Hz sine')
    ax.plot(dense_alias_t, np.sin(2*np.pi*folded_frequency*dense_alias_t), '--', label=f'{folded_frequency:g} Hz sine')
    ax.scatter(alias_t, source_values, color='black', label='Identical samples', zorder=3)
    ax.set(xlabel='Time (s)', ylabel='Signal value', title='Distinct curves, shared observations')
    ax.legend(); plt.show()
except NotImplementedError as error:
    print('Aliasing challenge is waiting for your implementation:', error)


Aliasing challenge is waiting for your implementation: alias_hz


### What can make reconstruction possible?
**Lecture companion: “Nyquist is a condition, not a repair tool” (slide 45) → “Interpolation adds a model between the dots” (slide 47)**

A **band-limited signal** contains no frequency components above a specified maximum. If a continuous-time signal contains no frequencies at or above half the sample rate, ideal uniform samples can uniquely specify it under the sampling theorem's assumptions. The half-rate boundary is the **Nyquist frequency**, $f_s/2$. At the boundary itself, phase matters: a sine may be sampled only at zero crossings. Use a strict margin below that boundary.

A filter can remove higher frequencies **before** sampling. Smoothing after aliases have already become indistinguishable cannot recover the missing information. Reducing the sample rate of an existing digital sequence requires the same care: filter before keeping every Dth value, using the new sample rate's boundary.

**Interpolation** estimates values between known observations using an assumed rule. A straight segment, holding the previous value, and ideal band-limited reconstruction are different rules. Drawing more points with interpolation is not the same as collecting more observations.


### Keep track of when observations occurred
**Lecture companion: “Timestamps are part of the signal” (slide 52)**

A **timestamp** records an observation's time. If intervals vary or observations are missing, the row index alone does not describe elapsed time. An array of values and an array of timestamps answer different questions.

We can estimate a uniform sequence from irregular observations with linear interpolation. It cannot justify filling an arbitrarily long gap. In this exercise we explicitly reject gaps longer than 0.08 seconds; that is a chosen rule for this example, not a universal safe gap.

### Coding challenge 2 · Interpolate after checking the data
Implement `resample_linear(t, X, target_t)` to interpolate each column of X at target times using `np.interp`. Return one row per target and preserve the number of columns. Reject non-increasing timestamps, nonfinite times or values, targets outside the observed interval, and gaps longer than 0.08 seconds by raising `ValueError`. Assume matching dimensions, at least two input rows, and nonempty target times.

**Predict:** which will change faster between nearby observations, a 2 Hz sine or a 12 Hz sine? Which is likely to have larger interpolation error with the same timestamp spacing? This exercise is separate from the basic sampling demonstration.


In [12]:
def resample_linear(t, X, target_t):
    # TODO: validate time coverage/gaps, then interpolate each signal column.
    raise NotImplementedError("resample_linear")


In [13]:
LAB_2_READY = False
try:
    probe_t = np.array([0., .02, .04])
    probe_x = np.column_stack([2*probe_t, 3*probe_t])
    np.testing.assert_allclose(resample_linear(probe_t, probe_x, np.array([.01,.03])), [[.02,.03],[.06,.09]])
    for bad_t, bad_x, target in [(np.array([0,.1,.12]), probe_x, np.array([.01])),
                                (np.array([0,.02,.02]), probe_x, np.array([.01])),
                                (probe_t, probe_x, np.array([-.01])),
                                (probe_t, np.full((3,2),np.nan), np.array([.01])),
                                (np.array([0,np.nan,.04]), probe_x, np.array([.01])),
                                (probe_t, probe_x, np.array([np.nan]))]:
        try: resample_linear(bad_t, bad_x, target)
        except ValueError: pass
        else: raise AssertionError('An invalid time/value case was not rejected')
    # EDIT: interpolation_frequency. Try 2, then 12, with unchanged timestamps.
    interpolation_frequency = 2.
    interpolation_rule = lambda t: 2 + 3*np.sin(2*np.pi*interpolation_frequency*t)
    rng = np.random.default_rng(120)
    t_irregular = np.arange(101)/50
    t_irregular[1:-1] += rng.uniform(-.004,.004,99)
    t_irregular = np.delete(t_irregular,[20,61])
    X_irregular = np.column_stack([interpolation_rule(t_irregular), np.sin(2*np.pi*3*t_irregular)])
    t_uniform = np.arange(100)/50
    interpolated = resample_linear(t_irregular, X_irregular, t_uniform)
    fig, axes = plt.subplots(1,2,figsize=(12,3.8),layout='constrained')
    axes[0].scatter(t_irregular, X_irregular[:,0], s=16, label='Observed values')
    axes[0].plot(t_uniform, interpolated[:,0], label='Uniform estimates')
    axes[0].set(xlabel='Time (s)', ylabel='Signal value', title='An assumed rule between observations')
    axes[1].plot(t_uniform, interpolated[:,0]-interpolation_rule(t_uniform))
    axes[1].set(xlabel='Time (s)', ylabel='Error in signal value', title='Interpolation is an estimate')
    axes[0].legend(fontsize=9); plt.show()
    print('First signal interpolation RMSE:', np.sqrt(np.mean((interpolated[:,0]-interpolation_rule(t_uniform))**2)))
    LAB_2_READY = True
except NotImplementedError as error:
    print('Interpolation challenge is waiting for your implementation:', error)


Interpolation challenge is waiting for your implementation: resample_linear


### Investigate and record
1. Sample `f` at 0.5-second intervals. Explain why a 2 Hz oscillation looks constant at these instants.
2. Repeat the interpolation experiment for a 12 Hz signal. Compare error with the 2 Hz case. Explain why the same timestamp jitter matters more for fast changes.
3. Predict then verify the alias of 27 Hz at 50 Hz. Show equality of the sampled sine waves, including phase/sign.

**Deliverable:** three functions, one aliasing demonstration, and an explanation of which information is missing inside a 0.2-second gap. **Extension:** compare interpolation before/after dropping every other sample; explain why anti-alias filtering must precede decimation.

<a id="bridge-2"></a>
### 2A. What acquisition preserves and loses
**Lecture companion: “Sampling and quantization form two independent axes” (slide 49) → “Quantization rounds; clipping loses the extremes” (slide 51)**

Sample rate determines observations per second; record duration determines how long we observe. More samples can result from either change. Faster sampling can capture higher frequencies with suitable anti-aliasing; a longer record can help separate nearby stable tones. Neither fills arbitrary gaps in knowledge of a continuous signal.

Sine repeats after whole multiples of $2\pi$, which explains why different frequencies can give the same samples. For real sines, phase and sign must also match. At exactly half the sample rate, a sine may land on all zero crossings while a cosine alternates between its extremes. This motivates a highest frequency strictly below $f_s/2$ and an anti-alias filter before acquisition.

Reducing a digital sample rate also creates a sampling problem: filter before retaining every Dth sample, using the new Nyquist frequency. Interpolation creates estimates under a model. Linear interpolation and holding the last value both pass through the measured samples while disagreeing between them. Ideal sinc reconstruction assumes a band-limited source and an idealized sampling setting; connecting dots with straight lines is not that theorem.

**Quantization** rounds amplitudes to levels. **Clipping** saturates them at a permitted range. A B-bit code has at most $2^B$ levels. Increasing the number of levels does not recover peaks lost to clipping. In the toy quantizer below, levels are integer multiples of a step and extend without a range limit; clipping is demonstrated as a separate operation.

### Coding challenge 2A
Implement `quantize_uniform(x, step)` by rounding to the nearest multiple of a positive, finite step. Preserve the input shape. Raise `ValueError` for a nonpositive or nonfinite step. NumPy's tie-to-even rounding is acceptable. This mathematical quantizer has no finite range limit; clipping is a separate operation below.

**Predict:** how should the maximum rounding error change when the step is halved? Can clipping to ±1 preserve a 1.4 peak, even with very fine quantization?


In [14]:
def quantize_uniform(x, step):
    # TODO: validate the step, then round values to its nearest integer multiple.
    raise NotImplementedError("quantize_uniform")


In [15]:
try:
    np.testing.assert_allclose(quantize_uniform(np.array([-.38, -.12, .12, .38]), .25), [-.5, 0, 0, .5])
    for bad_step in [0, -1, np.inf, np.nan]:
        try: quantize_uniform([1], bad_step)
        except ValueError: pass
        else: raise AssertionError('Invalid step was accepted')
    # EDIT: step_b. Keep the signal separate from the sampling lab variables.
    step_b = .25
    time_b = np.arange(301)/300
    signal_b = 1.4*np.sin(2*np.pi*2*time_b)
    rounded_b = quantize_uniform(signal_b, step_b)
    assert np.max(np.abs(rounded_b-signal_b)) <= step_b/2 + 1e-12
    fig, axs = plt.subplots(1, 2, figsize=(12,3), layout='constrained')
    for ax, values, title in zip(axs, [rounded_b, np.clip(signal_b, -1, 1)], ['Quantization', 'Clipping']):
        ax.plot(time_b, signal_b, '--', alpha=.5, label='Input')
        ax.plot(time_b, values, label='Output'); ax.legend()
        ax.set(xlabel='Time (s)', ylabel='Signal value', title=title)
    plt.show()
    print('Largest rounding error:', np.max(np.abs(rounded_b-signal_b)))
except NotImplementedError as error:
    print('Challenge 2A is waiting:', error)


Challenge 2A is waiting: quantize_uniform


<details><summary>Worked explanation · compare after predicting</summary>

Nearest-level rounding has error at most half a step, apart from floating-point roundoff. Halving the step halves that bound. Clipping maps all values above 1 to 1, so the lost peaks cannot be identified uniquely from the clipped record. Sampling, quantization, and clipping discard different kinds of information.

</details>

<a id="lab-3"></a>
## 3. LTI systems and convolution
**Lecture companion: “A system is a rule for changing a signal” (slide 56) → “Linearity and time invariance” (slide 62) → “Convolution follows from superposition” (slide 66)**

### Build intuition before the sum
A system is a rule such as “multiply every value by two” or “average the current value and the previous two.” For the sequence `[0, 0, 3, 0, 0]`, a three-sample causal average spreads the one-sample pulse over three outputs of height 1 (with zero prehistory). The system trades sharpness for reduced rapid variation.

**Predict:** sketch the same average applied to a step from 0 to 1. Does the output jump instantly or rise in stages? Can it change before the input changes? Use that sketch to interpret convolution below.

A system maps an input signal to an output. **Linearity** means $T(ax+bz)=aT(x)+bT(z)$. **Time invariance** means delaying the input delays the output by the same amount. A system with both properties is **linear time-invariant (LTI)**. A threshold is nonlinear; multiplying each sample by its time index is linear but time-varying.

The unit impulse $\delta[n]$ is one at zero and zero elsewhere. Any finite signal is a sum of shifted scaled impulses: $x[n]=\sum_k x[k]\delta[n-k]$. For an LTI system, each shifted impulse produces a shifted copy of $h=T(\delta)$; superposition gives **convolution**:

$$y[n]=(x*h)[n]=\sum_k x[k]h[n-k].$$

The **impulse response** h is the output caused by a unit impulse. If it has only finitely many nonzero terms, the resulting convolution system is a **finite impulse response (FIR) filter**. Its fixed taps are those terms. It is linear and time-invariant.

Full convolution of lengths N and M has N+M−1 outputs under the assumption that both inputs are zero elsewhere. For live filtering with causal taps `h[0], h[1], ...`, the output at n uses the present and past. Zero startup history and complete-window output are distinct boundary conventions.

### Implement
- `convolve_full(x, h)` for real 1D inputs **from the sum**, without `np.convolve`, FFTs, or correlation helpers.
- `fir_valid(X, h)` for an N×C record: apply the same causal filter separately to each channel and return only complete-window outputs, shape `(N-M+1, C)`. Reuse your convolution function.

Use asymmetric kernels in your checks. A symmetric moving average cannot reveal whether you accidentally implemented correlation instead of convolution.

In [16]:
def convolve_full(x, h):
    # TODO: implement the discrete convolution sum with zero extension.
    raise NotImplementedError("convolve_full")

def fir_valid(X, h):
    # TODO: apply causal taps to each channel; retain complete-window outputs.
    raise NotImplementedError("fir_valid")

In [17]:
FS_3 = 50  # independent rate for the filtering experiment
LAB_3_READY = False
try:
    np.testing.assert_allclose(convolve_full([1,2],[1,3]),[1,5,6])
    np.testing.assert_allclose(convolve_full([0,1,0],[1,2,4]),[0,1,2,4,0])
    probe=np.column_stack([np.arange(6.),-np.arange(6.)])
    np.testing.assert_allclose(fir_valid(probe,np.array([1.,2.])),[[1,-1],[4,-4],[7,-7],[10,-10],[13,-13]])
    x=np.array([1.,-2,3,0]);z=np.array([2.,0,1,-1]);h=np.array([.2,.5,.3])
    np.testing.assert_allclose(convolve_full(2*x-3*z,h),2*convolve_full(x,h)-3*convolve_full(z,h))
    np.testing.assert_allclose(convolve_full(np.r_[0,0,x],h),np.r_[0,0,convolve_full(x,h)])
    t=np.arange(200)/FS_3
    rng=np.random.default_rng(3)
    clean=np.exp(-.5*((t-1.5)/.10)**2)
    noisy=clean+rng.normal(0,.15,len(t))
    fig,ax=plt.subplots()
    ax.plot(t,noisy,alpha=.35,label='Noisy brief event')
    ax.plot(t,clean,'k--',label='True event')
    for w in [1,5,21]:
        y=fir_valid(noisy[:,None],np.ones(w)/w)[:,0]
        ax.plot(t[w-1:],y,label=f'{w} taps; window-center lag {(w-1)/(2*FS_3)*1000:.0f} ms')
    ax.set(xlabel='Time (s)',ylabel='Signal value',title='Complete windows keep the causal time alignment')
    ax.legend(fontsize=9);plt.show()
    # PROVIDED intuition check: the same rule on a pulse and a step.
    idx3 = np.arange(10)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3), layout='constrained')
    for ax, source, label in zip(axes, [3*(idx3 == 2).astype(float), (idx3 >= 2).astype(float)], ['Pulse', 'Step']):
        result = convolve_full(source, np.ones(3)/3)[:len(source)]
        ax.step(idx3, source, where='post', linestyle='--', label='Input')
        ax.plot(idx3, result, 'o-', label='3-point causal average')
        ax.set(xlabel='Sample index', ylabel='Signal value', title=label + ': zero prehistory')
        ax.legend(fontsize=9)
    plt.show()
    print('Asymmetric-kernel, impulse, linearity, and shift tests passed.')
    LAB_3_READY = True
except NotImplementedError as error:
    print("Lab 3 is waiting for your implementation:", error)

Lab 3 is waiting for your implementation: convolve_full


### Investigate and record
1. Replace the averaging kernel with `[1, -1]`. What changes does this emphasize? What happens to independent noise?
2. Demonstrate that `T(x) = (x >= 0)` fails superposition using a numerical counterexample.
3. Compare the complete-window startup time and window-center lag for 5 and 21 taps at 50 Hz. Do not shift a causal output backward just to make curves line up.

**Deliverable:** your convolution and multichannel FIR functions, one nonlinear counterexample, and a filter choice justified by peak preservation and delay. **Before moving on:** which errors can a smoother fix, and which require better acquisition or a different model?

<a id="bridge-3"></a>
### 3A. Systems, boundaries, and convolution one sample at a time
**Lecture companion: “Convolution: fix one output index” (slide 67) → “Full, valid, and centered output have different meanings” (slide 70) → “Averaging reduces independent noise in a predictable way” (slide 74)**

A memoryless system uses only the current input. A causal system uses no future input; it may use a long past history. Thus a delay has memory and is causal, while a centered average usually has memory and is noncausal. Offline processing can access saved future samples. “Same length” output says nothing about causality.

Linearity and time invariance are separate tests. `T(x)=x+1` is not linear because it maps zero to one. `T(x)[n]=n*x[n]` is linear but changes with absolute time. Bounded-input, bounded-output stability asks whether every bounded input has a bounded output. A finite moving average is stable; a running sum driven by ones grows without bound. For an LTI system, an absolutely summable impulse response guarantees this stability.

To find a single output $y[n]=\sum_k x[k]h[n-k]$, **hold n fixed**, sweep k, form each product, and sum. As a function of k, $h[n-k]$ is a reversed and shifted kernel. With `x=[1,2,1]` and `h=[1,3]`, the outputs are `[1,5,7,3]`. Do not confuse the output index n with the summation index k.

Full convolution zero-extends both finite inputs. Valid convolution retains only complete overlap. Zero-padding, reflection, and periodic wraparound are different assumptions about unseen values. We will return to the consequences of periodic wraparound after introducing Fourier analysis in Lab 4.

An **offset** is a constant added level; **drift** is a slowly changing level. **Noise** is variation unwanted for the question we are asking; it need not be random. **Interference** is an unwanted component with a separate source, often periodic. These need different interpretations. Averaging M independent zero-mean noise samples with variance $\sigma^2$ gives variance $\sigma^2/M$ and standard deviation $\sigma/\sqrt M$. Correlation changes that result; constant offset passes through an average. Low-pass, high-pass, band-pass, and notch describe frequency-selective gain shapes. Filter choice also affects timing and desired changes.

### Coding challenge 3A
Implement `convolution_terms(x, h, n)` returning one term for each input index k: `x[k]*h[n-k]`, treating h as zero outside its valid indices. Return a length-len(x) float array. Do not call a convolution function. The demo adds the terms and compares with your Lab 3 function when available.

**Predict:** write all three terms for n=0, 1, 2, and 3 before running. Then explain why the 5 in the second output comes from two contributions.


In [18]:
def convolution_terms(x, h, n):
    # TODO: hold n fixed, compute the shifted-kernel products over k, and zero invalid indices.
    raise NotImplementedError("convolution_terms")


In [19]:
try:
    input_c, kernel_c = np.array([1.,2,1]), np.array([1.,3])
    terms_c = np.stack([convolution_terms(input_c, kernel_c, n) for n in range(4)])
    np.testing.assert_allclose(terms_c, [[1,0,0], [3,2,0], [0,6,1], [0,0,3]])
    output_c = terms_c.sum(axis=1)
    np.testing.assert_allclose(output_c, [1,5,7,3])
    if globals().get('LAB_3_READY', False):
        np.testing.assert_allclose(output_c, convolve_full(input_c, kernel_c))
    fig, axs = plt.subplots(1, 4, figsize=(12,3), layout='constrained')
    for n, ax in enumerate(axs):
        ax.bar(np.arange(3), terms_c[n]); ax.set(xlabel='Input index k', ylabel='Contribution', title=f'n={n}: sum={output_c[n]:.0f}', ylim=(0,7), xticks=[0,1,2])
    plt.show()
    # PROVIDED statistical experiment. Independent blocks avoid overlapping outputs.
    noise_c = np.random.default_rng(300).normal(0, 1, (20000, 25))
    print('M | measured std | predicted std')
    for width_c in [1,4,25]:
        print(width_c, round(noise_c[:,:width_c].mean(axis=1).std(),3), round(1/np.sqrt(width_c),3))
except NotImplementedError as error:
    print('Challenge 3A is waiting:', error)


Challenge 3A is waiting: convolution_terms


<details><summary>Worked explanation · compare after predicting</summary>

The terms are `[1,0,0]`, `[3,2,0]`, `[0,6,1]`, and `[0,0,3]`. Summing over k produces `[1,5,7,3]`. The kernel index is n−k, so its position changes for every output. The independent-noise experiment should approach standard deviations 1, 0.5, and 0.2; finite random draws need not match exactly.

</details>

<a id="lab-4"></a>
## 4. Fourier analysis: projections and system response
**Lecture companion: “From time to frequency” (slide 78) → “Complex numbers are pairs with a geometric meaning” (slide 82) → “The DFT is a set of projections” (slide 88) → “An LTI filter acts on each frequency” (slide 94)**

### Hear the idea before reading the transform
A sum of a slow and a fast sine can look complicated in time. A frequency description lists how much of each repeating pattern is present. Imagine matching the record against candidate sine/cosine pairs: a matching rhythm accumulates a large inner product; a mismatched rhythm tends to cancel over complete cycles.

**Predict:** sketch the spectrum of a 2 Hz sine, then of that sine plus a weaker 12 Hz component. What should doubling only the fast component change? What should delaying the entire record change? The mathematical transform below makes those questions precise.

A complex number $z=a+jb$ stores two real coordinates, with $j^2=-1$. Its magnitude is $\sqrt{a^2+b^2}$ and its angle is measured from the positive real axis. A point moving around the unit circle has horizontal coordinate cosine and vertical coordinate sine. Euler's identity $e^{j\theta}=\cos\theta+j\sin\theta$ packages two quadrature components into one complex number. A Fourier coefficient is an inner product with a complex sinusoid; its magnitude and angle encode strength and phase relative to that basis.

For N samples, the **discrete Fourier transform (DFT)** is $X[k]=\sum_{n=0}^{N-1}x[n]e^{-j2\pi kn/N}$, with bin frequency $f_k=kf_s/N$. The inverse divides by N. The **fast Fourier transform (FFT)** computes the same transform faster; implementing a small direct DFT reveals the projections rather than treating it as a black box.

For a causal FIR filter, complex exponentials retain their frequency and are multiplied by $H(\omega)=\sum_m h[m]e^{-j\omega m}$. Thus convolution in time corresponds to multiplication in frequency (with appropriate zero-padding for finite linear convolution). The **gain** is the output-to-input amplitude ratio. Filter phase changes timing; magnitude gives gain. For a symmetric M-tap average, the phase slope corresponds to a **group delay** of $(M-1)/2$ sample intervals away from response zeros. This formalizes the window-center lag observed in Lab 3.

Finite records implicitly window the signal. A tone not centered on a DFT bin spreads across bins. A Hann window reduces distant leakage while widening the main lobe. Zero-padding samples the spectrum more densely; it does not create new resolving power.

### Implement
- `dft_direct(x)` → all N complex DFT coefficients; use the definition, no FFT call. Test at N=64.
- `fir_response(h, frequencies, fs)` → complex H at each supplied frequency in Hz; no FFT call.

The plotting cell supplies a correctly scaled one-sided magnitude spectrum for an even-length record. A sinusoid at DC or Nyquist needs different scaling from interior bins.

In [20]:
def dft_direct(x):
    # TODO: complex inner products with N discrete Fourier basis vectors.
    raise NotImplementedError("dft_direct")

def fir_response(h, frequencies, fs):
    # TODO: evaluate the FIR frequency-response sum at frequencies in Hz.
    raise NotImplementedError("fir_response")

In [21]:
FS_4 = 50  # independent rate for the frequency-response experiment
LAB_4_READY = False
try:
    need(globals().get('LAB_3_READY',False),'finish Lab 3 for the time-domain filter comparison')
    n=np.arange(64)
    x=3*np.cos(2*np.pi*8*n/64+.4)+1.5*np.sin(2*np.pi*15*n/64)
    D=dft_direct(x)
    np.testing.assert_allclose(D,np.fft.fft(x),atol=1e-10)
    np.testing.assert_allclose(np.abs(D[8])*2/64,3,atol=1e-10)
    np.testing.assert_allclose(np.angle(D[8]),.4,atol=1e-10)
    freq=np.linspace(0,25,501);h=np.ones(5)/5
    H=fir_response(h,freq,FS_4)
    np.testing.assert_allclose(fir_response([1],freq,FS_4),1)
    np.testing.assert_allclose(fir_response(h,np.array([0,10]),FS_4),[1,0],atol=1e-12)
    # Supply prehistory so the 4-second output has no startup transient.
    t_history=np.arange(-4,200)/FS_4
    mixed=3*np.sin(2*np.pi*2*t_history)+1.5*np.sin(2*np.pi*12*t_history)
    y=fir_valid(mixed[:,None],h)[:,0]
    f=np.fft.rfftfreq(200,1/FS_4)
    amplitude=2*np.abs(np.fft.rfft(y))/200;amplitude[[0,-1]]/=2
    expected=3*abs(fir_response(h,np.array([2]),FS_4)[0])
    np.testing.assert_allclose(amplitude[8],expected,atol=1e-10)
    fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    axes[0].plot(freq,np.abs(H));axes[0].set(xlabel='Frequency (Hz)',ylabel='Gain',title='Five-tap moving average')
    axes[1].plot(f,amplitude);axes[1].set(xlabel='Frequency (Hz)',ylabel='Signal amplitude',title='Predicted gains match the filtered record')
    plt.show()
    print(f'2 Hz gain: {abs(fir_response(h,np.array([2]),FS_4)[0]):.3f}')
    print(f'12 Hz gain: {abs(fir_response(h,np.array([12]),FS_4)[0]):.3f}')
    LAB_4_READY = True
except NotImplementedError as error:
    print("Lab 4 is waiting for your implementation:", error)

Lab 4 is waiting for your implementation: finish Lab 3 for the time-domain filter comparison


### Investigate and record
1. For windows of 5 and 21 samples, compute gain at 2 Hz and 12 Hz. Decide whether stronger suppression of the fast component is worth the loss of the desired 2 Hz component.
2. Replace the bin-centered tone in a 4-second record with 11.3 Hz. Compare rectangular and Hann-windowed spectra. Use `2*abs(rfft(x*w))/sum(w)` for interior-bin amplitude display; even with this scaling, an off-bin tone need not reach its true amplitude at any single bin.
3. Add zeros to the same record before the FFT. Then acquire a longer modeled record instead. Explain which changes bin spacing and which adds information about closely spaced tones.

**Deliverable:** two functions, a predicted-versus-measured gain check, and a short explanation of why an FFT peak alone does not identify the physical source. **Extension:** implement inverse DFT and reconstruct the signal; inspect what removing selected coefficients does in time.

<a id="bridge-4"></a>
### 4A. Complex geometry, spectra, and reconstruction
**Lecture companion: “A one-sided amplitude spectrum needs endpoint rules” (slide 90) → “The inverse transform adds the components back” (slide 95) → “Magnitude and phase answer different questions” (slide 86)**

A complex number $z=a+jb$ is a pair of real coordinates, with $j^2=-1$, magnitude $\sqrt{a^2+b^2}$, and angle `atan2(b,a)`. A point on the unit circle has coordinates cos θ and sin θ. Rotating it at constant angular speed creates those two oscillations. Multiplication by j rotates it by 90 degrees. Positive and negative frequencies correspond to opposite rotation directions.

A dot product with a matching oscillation accumulates; mismatched DFT basis oscillations cancel over the full record. Sine and cosine together detect any phase. A real cosine combines two oppositely rotating complex exponentials. For real input, DFT coefficients obey conjugate symmetry, $X[N-k]=X[k]^*$: paired bins have equal magnitudes and opposite phases.

Fourier series describe harmonics of periodic signals. A continuous Fourier transform has continuous frequency. The DTFT describes a discrete-time sequence with continuous periodic frequency. The DFT uses N frequency bins for N samples under a periodic extension. The FFT computes the DFT efficiently. The inverse convention here is $x[n]=\frac1N\sum_k X[k]e^{j2\pi kn/N}$.

Magnitude says how strong a component is; phase says where it is in its cycle relative to the time origin. A circular shift preserves DFT magnitudes while changing phases. Arbitrarily resetting phase usually changes the waveform. Integer multiples of a fundamental are harmonics; combining them can build nonsinusoidal periodic shapes such as square-wave approximations.

The DFT bin spacing is $f_s/N$. More zero-padding adds plotted frequency samples without extending the observation. A longer record adds information about nearby stable tones. Finite-record endpoints and the periodic extension explain leakage. A taper reduces distant leakage while broadening peaks. Magnitude scaling, window choice, and record duration must be stated when interpreting a spectrum.

### Coding challenges 4A and 4B
- `one_sided_amplitude(x, fs)` returns `(frequencies, amplitudes)` for a real 1D record of length at least 2 and positive fs. Use `rfft` and `rfftfreq`. Normalize by N, double only bins with distinct negative-frequency partners. DC is never doubled; for even N, Nyquist is not doubled; for odd N, the last retained bin is doubled. No taper is applied in this function.
- `idft_direct(coefficients)` implements the inverse DFT sum without an inverse FFT. Return all N reconstructed complex samples. Tiny imaginary parts may remain for real-source data.

**Predict:** for 64 samples at 64 Hz of `1 + 3*cos(2*pi*8*n/64) + .7*cos(pi*n)`, identify the three amplitude peaks. Why must the last one not be doubled? What would happen to a pulse if you erased all its spectral phases?


In [22]:
def one_sided_amplitude(x, fs):
    # TODO: build the frequency axis and scale interior bins without doubling DC/Nyquist.
    raise NotImplementedError("one_sided_amplitude")

def idft_direct(coefficients):
    # TODO: add the positive-exponent complex basis functions and divide by N.
    raise NotImplementedError("idft_direct")


In [23]:
try:
    index_d = np.arange(64)
    signal_d = 1 + 3*np.cos(2*np.pi*8*index_d/64) + .7*np.cos(np.pi*index_d)
    freq_d, amp_d = one_sided_amplitude(signal_d, 64)
    np.testing.assert_allclose(freq_d[[0,8,32]], [0,8,32])
    np.testing.assert_allclose(amp_d[[0,8,32]], [1,3,.7], atol=1e-12)
    odd_d = 2*np.cos(2*np.pi*32*np.arange(65)/65)
    np.testing.assert_allclose(one_sided_amplitude(odd_d, 65)[1][-1], 2, atol=1e-12)
    np.testing.assert_allclose(idft_direct(np.fft.fft(signal_d)), signal_d, atol=1e-11)
    pulse_d = np.exp(-.5*((index_d-18)/3)**2)
    coefficients_d = np.fft.fft(pulse_d)
    rebuilt_d = idft_direct(coefficients_d)
    erased_d = idft_direct(np.abs(coefficients_d))
    np.testing.assert_allclose(rebuilt_d.real, pulse_d, atol=1e-12)
    fig, axs = plt.subplots(1, 2, figsize=(12,3), layout='constrained')
    axs[0].stem(freq_d, amp_d, basefmt=' '); axs[0].set(xlabel='Frequency (Hz)', ylabel='Signal amplitude', title='DC, interior, and Nyquist bins')
    axs[1].plot(index_d, pulse_d, label='Original'); axs[1].plot(index_d, rebuilt_d.real, '--', label='Full inverse'); axs[1].plot(index_d, erased_d.real, label='Phases erased')
    axs[1].set(xlabel='Sample index', ylabel='Signal value', title='Magnitude alone loses timing'); axs[1].legend(fontsize=9)
    plt.show()
    print('Even/odd endpoint scaling and inverse reconstruction checks passed.')
except NotImplementedError as error:
    print('Challenges 4A–4B are waiting:', error)


Challenges 4A–4B are waiting: one_sided_amplitude


<details><summary>Worked explanation · compare after predicting</summary>

The amplitude peaks are 1 at DC, 3 at 8 Hz, and 0.7 at Nyquist (32 Hz). DC and even-length Nyquist bins are self-paired, so doubling them overcounts. Odd N has no exact Nyquist bin, so its last retained bin still has a partner. The full complex coefficients reconstruct the pulse; magnitudes with erased phases produce a different time waveform. A pure delay has unit gain and a phase slope; amplitude alone cannot reveal that delay.

</details>

<a id="part-ii"></a>
## Fundamentals checkpoint: explain before applying
**Lecture companion: “Fundamentals checkpoint” (slide 102) → “Part II · Applying signals to GuaqBand” (slide 103)**

Close the code for a moment and answer from a sketch or a sentence:

1. What distinguishes changing amplitude, adding an offset, and delaying a signal?
2. Why can a continuous-time signal and a sampled sequence contain different information? Can smoothing undo aliasing?
3. Why does a causal moving average reduce rapid variation and also round a short pulse?
4. How do a system's impulse response and frequency response describe the same operation in different ways?
5. What does a Fourier peak tell you, and what does it leave unknown?

<details><summary>Compare explanations</summary>

Scaling changes vertical excursions, offset moves the centerline, and delay shifts timing. Sampling retains selected instants; distinct continuous signals can agree there, so later smoothing cannot distinguish them. Averaging neighboring values cancels some rapid oscillations but spreads brief changes through time. For an LTI system, the impulse response determines convolution, and its Fourier transform gives the amplitude gain and phase shift for every frequency. A Fourier peak describes oscillatory content in a finite record; it does not identify the source or establish usefulness.

</details>

## Part II · Apply the fundamentals to GuaqBand
Everything above can be studied independently of this project. We now introduce a specific measurement and ask which signal properties the application should preserve.

GuaqBand has four magnetometers with three axes each: **12 magnetic-field channels in microteslas**. A frame holds their near-synchronous readings at a target 50 Hz. Magnet motion changes the field; the gesture label is an inference from those changes.

| Fundamental idea | Application question |
|---|---|
| Samples and timestamps | Are frames uniformly spaced? What happened during gaps? |
| Offset, noise, and drift | Which field level should count as relaxed? |
| Causal filtering | How much variation and delay can a live decision tolerate? |
| Frequency response | Which components will the selected smoother attenuate? |
| Mean and RMS | What information survives the summary of a window? |

### Supplied project data
Only now do we generate **48 synthetic 12-channel trials**, 12 each for `rest`, `wrist_up`, `spread`, and `fist`. Each trial has 200 samples at 50 Hz with channels `s0x, s0y, s0z, ..., s3z`. The first second is relaxed; the intended gesture occurs around 1.4–3.3 s. Trials vary in offset, gain, noise, drift, and timing.

These are teaching simulations, **not wearer recordings or a validated magnetic-field model**. Train/tune/test assignment uses whole trials before any windows are extracted. High synthetic accuracy does not establish real performance. The application reuses the mathematical operations you explored in Part I.


In [24]:
FS = 50
CLASSES = np.array(["rest", "wrist_up", "spread", "fist"])
CHANNELS = [f"s{s}{axis}" for s in range(4) for axis in "xyz"]

# Distinct, partially overlapping channel patterns chosen for teaching.
PATTERNS = np.array([
    [0,0,0, 0,0,0, 0,0,0, 0,0,0],
    [7,2,-3, 6,1,-2, 3,-2,1, 2,-3,1],
    [2,6,3, -2,5,2, 3,6,-2, -2,4,-3],
    [-5,2,6, -4,2,5, -5,-2,6, -4,-1,5]], dtype=float)

def make_trials(seed=66):
    rng = np.random.default_rng(seed)
    t = np.arange(200) / FS
    trials = []
    for label in range(4):
        for rep in range(12):
            onset = 1.4 + rng.uniform(-0.08, 0.08)
            release = 3.25 + rng.uniform(-0.08, 0.08)
            envelope = np.clip((t-onset)/0.20, 0, 1) * np.clip((release-t)/0.20, 0, 1)
            gain = rng.uniform(0.7, 1.3)
            offset = rng.normal(40, 8, 12)
            pattern = PATTERNS[label] + (rng.normal(0, .8, 12) if label else 0)
            drift = t[:, None] * rng.normal(0, .22, (1,12))
            shared_noise = rng.normal(0, .45, (len(t),1))
            noise = rng.normal(0, 1.2, (len(t),12)) + shared_noise
            ripple = .6*np.sin(2*np.pi*12*t + rng.uniform(0, 2*np.pi))[:,None]
            clean = envelope[:, None] * gain * pattern[None,:]
            x = offset + clean + drift + noise + ripple
            trials.append(dict(id=f"{CLASSES[label]}-{rep:02d}", label=label, t=t.copy(),
                               x=x, clean=clean, baseline=offset,
                               split="train" if rep < 6 else "tune" if rep < 9 else "test"))
    return trials

TRIALS = make_trials()
assert len(TRIALS) == 48 and TRIALS[0]["x"].shape == (200,12)

def show_confusion(matrix, title):
    fig, ax = plt.subplots(figsize=(6,5), layout="constrained")
    ax.imshow(matrix, cmap="Blues", vmin=0)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, str(matrix[i,j]), ha="center", va="center",
                    color="white" if matrix[i,j] > matrix.max()/2 else "black")
    ax.set(xticks=range(4), xticklabels=CLASSES, yticks=range(4), yticklabels=CLASSES,
           xlabel="Predicted gesture", ylabel="True gesture", title=title)
    plt.show()

print(f"{len(TRIALS)} trials × 200 samples × 12 channels. All data are synthetic.")

48 trials × 200 samples × 12 channels. All data are synthetic.


<a id="lab-5"></a>
## 5. A multichannel processing pipeline
**Lecture companion: “5 · Multichannel processing” (slide 106) → “A feature vector is deliberate information loss” (slide 108) → “Lab 5 · Extract meaningful features” (slide 111)**

GuaqBand observes 12 field components. Calibration estimates a vector baseline $b\in\mathbb{R}^{12}$ from a known relaxed interval. For each channel we smooth a short history, subtract its baseline, then summarize a window. A baseline is a physical reference estimate; subtracting a global mean across gestures can erase part of what we want to detect.

A window's signed mean retains the direction of a change. RMS retains magnitude but loses sign. Together they produce 24 features: 12 means followed by 12 RMS values. Feature extraction compresses a time series; different waveforms can have the same summary. Features appropriate for held poses may fail for dynamic motions.

The repository uses 19 raw frames, a five-sample causal average, and 15 resulting samples. At 50 Hz, 19 frames span **0.36 s from first timestamp to last**, while successive batches of 19 sample intervals take 0.38 s. Five-tap smoothing has nominal group delay 0.04 s. Window duration, startup history, and computational time are different contributors to latency.

### Implement
- `estimate_baseline(X, rest_mask)` → a length-12 vector, averaging only relaxed rows.
- `window_features(raw, baseline, taps)` → smooth with `fir_valid`, subtract baseline, return signed means followed by RMS values. Support alternative tap lengths on the same raw window for comparisons.
- `standardize_fit(F)` → `(mu, scale)` per feature, with zero standard deviations replaced by 1.
- `standardize_apply(F, mu, scale)` → transformed values. Use only training statistics, including when transforming tuning/test features.

In [25]:
def estimate_baseline(X, rest_mask):
    # TODO: estimate one relaxed level per channel.
    raise NotImplementedError("estimate_baseline")

def window_features(raw, baseline, taps):
    # TODO: reuse your FIR, then summarize signed changes and magnitudes.
    raise NotImplementedError("window_features")

def standardize_fit(F):
    # TODO: fit feature statistics using the supplied training rows only.
    raise NotImplementedError("standardize_fit")

def standardize_apply(F, mu, scale):
    # TODO: transform data with saved statistics.
    raise NotImplementedError("standardize_apply")

In [26]:
LAB_5_READY = False
try:
    need(globals().get('LAB_3_READY',False),'finish Lab 3: fir_valid')
    b=estimate_baseline(np.array([[1.,2.],[3.,4.],[99.,99.]]),np.array([True,True,False]))
    np.testing.assert_allclose(b,[2,3])
    raw=np.tile(np.arange(12.),(19,1))
    feat=window_features(raw,np.ones(12),np.ones(5)/5)
    np.testing.assert_allclose(feat,np.r_[np.arange(12)-1,np.abs(np.arange(12)-1)],atol=1e-12)
    mu,scale=standardize_fit(np.array([[1.,2.],[1.,4.]]))
    np.testing.assert_allclose(standardize_apply(np.array([[1.,6.]]),mu,scale),[[0,3]])
    assert scale[0]==1
    trial=TRIALS[12]
    baseline=estimate_baseline(trial['x'],trial['t']<1)
    features=window_features(trial['x'][100:119],baseline,np.ones(5)/5)
    assert features.shape==(24,)
    fig,ax=plt.subplots()
    ax.plot(range(12),features[:12],'o-',label='Signed mean')
    ax.plot(range(12),features[12:],'s--',label='RMS')
    ax.set(xlabel='Channel index',ylabel='Feature value (µT)',title='Features preserve different aspects of a held pose')
    ax.legend();plt.show()
    print('24 features:',np.round(features,2))
    LAB_5_READY = True
except NotImplementedError as error:
    print("Lab 5 is waiting for your implementation:", error)

Lab 5 is waiting for your implementation: finish Lab 3: fir_valid


### Investigate and record
1. Construct two 19×12 windows with opposite constant changes relative to the same baseline. Show that their RMS features match but their signed means differ.
2. Estimate the baseline from a relaxed interval, then from an interval including a press. Quantify how the features change.
3. Create two different waveforms with the same mean and RMS. Explain one kind of movement the feature vector cannot distinguish.

**Deliverable:** four functions and a feature-loss example. **Extension:** add a derivative RMS feature or channel correlations, but decide what behavior it should capture before measuring classification accuracy.

<a id="lab-6"></a>
## 6. GuaqBand capstone: evaluate a whole pipeline
**Lecture companion: “6 · Evaluate the complete pipeline” (slide 112) → “Split by trial before extracting windows” (slide 114) → “Lab 6 · Build and defend the pipeline” (slide 116)**

A classifier maps feature vectors to labels. Here, 3-nearest-neighbor classification chooses the most common label among the three closest standardized training examples. This is a simple use of geometry, not a guarantee of physiological meaning.

Evaluation must respect how data were acquired. Overlapping windows from one trial are highly related. If some enter training and others testing, the result can overstate performance on a new trial. Keep **all windows from each trial in one split**, fit scales on training data only, choose processing settings on tuning data, and inspect the test result after the choice is fixed.

### Implement
- `knn_predict(train_F, train_y, query_F, k=3)` → integer predictions. Use Euclidean distance. Break equal distances by training-row order, and vote ties by the nearest tied class.
- `confusion_counts(y_true, y_pred)` → a 4×4 count matrix, rows true labels and columns predictions.

The supplied assembly code uses your baseline, filter, feature, scaling, and classifier functions. It creates eight overlapping windows per trial in the held interval. Compare 1-, 5-, and 11-tap averages using **tuning** accuracy. Ties choose the shorter filter to avoid extra delay. The final cell evaluates the chosen filter on untouched test trials, then on a deliberate change in sensor gain.

A separate baseline is available for every trial because its initial second is relaxed. This uses acquisition metadata, not test gesture labels. It assumes a comparable rest calibration is available at deployment.

In [27]:
def knn_predict(train_F, train_y, query_F, k=3):
    # TODO: distances, nearest neighbors, voting, and deterministic tie handling.
    raise NotImplementedError("knn_predict")

def confusion_counts(y_true, y_pred):
    # TODO: accumulate true/predicted counts for the four classes.
    raise NotImplementedError("confusion_counts")

In [28]:
# PROVIDED ASSEMBLY: every processing operation calls a function you implemented.
def assemble(trials, taps, shift=False):
    F=[]; labels=[]; groups=[]; splits=[]
    for trial in trials:
        X=trial['x'].copy()
        # Stress test: altered per-axis response after calibration; still synthetic.
        if shift:
            X=trial['baseline']+(X-trial['baseline'])*np.tile([.25,1.8,-.6],4)
        baseline=estimate_baseline(X,trial['t']<1)
        for start in range(90,146,7):
            F.append(window_features(X[start:start+19],baseline,taps))
            labels.append(trial['label']);groups.append(trial['id']);splits.append(trial['split'])
    return np.array(F),np.array(labels),np.array(groups),np.array(splits)

def fit_and_predict(F,y,splits,query_mask):
    train=splits=='train'
    mu,scale=standardize_fit(F[train])
    train_F=standardize_apply(F[train],mu,scale)
    query_F=standardize_apply(F[query_mask],mu,scale)
    return knn_predict(train_F,y[train],query_F),mu,scale,train_F

In [29]:
LAB_6_READY = False
try:
    need(globals().get('LAB_5_READY',False),'finish Lab 5 and its Lab 3 dependency')
    np.testing.assert_array_equal(knn_predict(np.array([[0.],[1.],[2.]]),np.array([2,1,0]),np.array([[.1]]),3),[2])
    np.testing.assert_array_equal(knn_predict(np.array([[0.],[0.],[2.]]),np.array([3,1,0]),np.array([[0.]]),1),[3])
    np.testing.assert_array_equal(confusion_counts([0,1,1],[0,0,1])[:2,:2],[[1,0],[1,1]])
    candidates=[]
    for w in [1,5,11]:
        F,y,groups,splits=assemble(TRIALS,np.ones(w)/w)
        train_groups=set(groups[splits=='train']);tune_groups=set(groups[splits=='tune']);test_groups=set(groups[splits=='test'])
        assert not (train_groups&tune_groups or train_groups&test_groups or tune_groups&test_groups)
        tune=splits=='tune'
        predictions,_,_,_=fit_and_predict(F,y,splits,tune)
        accuracy=np.mean(predictions==y[tune])
        candidates.append((accuracy,w))
        print(f'{w:2d} taps | tuning window accuracy {accuracy:.1%} | nominal FIR delay {(w-1)/(2*FS)*1000:.0f} ms')
    chosen_w=max(candidates,key=lambda pair:(pair[0],-pair[1]))[1]
    print('Chosen before test evaluation:',chosen_w,'taps')
    F,y,groups,splits=assemble(TRIALS,np.ones(chosen_w)/chosen_w)
    test=splits=='test'
    predictions,mu,scale,train_F=fit_and_predict(F,y,splits,test)
    cm=confusion_counts(y[test],predictions)
    show_confusion(cm,'Held-out synthetic trials: window counts')
    print(f'Test window accuracy: {np.mean(predictions==y[test]):.1%} ({test.sum()} windows from {len(set(groups[test]))} trials)')
    # Trial-level majority votes avoid pretending overlapping windows are independent trials.
    trial_results=[]
    for group in sorted(set(groups[test])):
        mask=groups[test]==group
        vote=np.bincount(predictions[mask],minlength=4).argmax()
        trial_results.append(vote==y[test][mask][0])
    print(f'Test trial majority-vote accuracy: {np.mean(trial_results):.1%}')
    shifted,_,_,_=assemble(TRIALS,np.ones(chosen_w)/chosen_w,shift=True)
    shift_predictions=knn_predict(train_F,y[splits=='train'],standardize_apply(shifted[test],mu,scale))
    print(f'Altered sensor response, same trained model: {np.mean(shift_predictions==y[test]):.1%}')
    LAB_6_READY = True
except NotImplementedError as error:
    print("Lab 6 is waiting for your implementation:", error)

Lab 6 is waiting for your implementation: finish Lab 5 and its Lab 3 dependency


### Capstone report
Write a short engineering recommendation containing:
- The filter you selected, tuning results, and the delay you accepted.
- Test confusion counts and accuracy, naming **windows versus trials** as the unit of evaluation.
- One error mode revealed by the changed sensor response and one acquisition experiment you would run next.
- Why this synthetic result cannot establish accuracy for new wearers, placements, or real sensor noise.

**Core acceptance:** the whole pipeline runs with your functions; splits are disjoint by trial; scales come from training only; filter selection precedes test inspection; your report connects errors to signal-processing choices. There is no required accuracy score.

**Optional extensions:** compare means-only versus RMS-only features; implement a streaming two-prediction stabilizer and measure onset delay; compare a random-window split with the trial split and explain why overlapping windows create dependence. Do not optimize repeatedly against the test set.

## Optional hardware extension: replace the modeled record
The GuaqBand finger stream uses `FRAME,sequence,device_us,s0x,s0y,s0z,...,s3z`. It has four three-axis magnetometers, target rate 50 Hz, and sequential sensor reads grouped into near-synchronous frames. This extension targets the magnetic `FRAME` packets; `MOTION` packets from the BNO085 are a different signal source.

1. Upload the existing GuaqBand firmware and close any other serial monitor. Record one person's relaxed baseline and several labeled gestures, retaining sequence numbers and device timestamps. Keep the first second of each trial relaxed, matching the lab protocol.
2. Use the optional recorder below (`python -m pip install pyserial` once). Set the port and `RECORD_HARDWARE = True` only when ready. It writes a raw text file and stops after 6 seconds. Label and record each trial separately; do not overwrite files from different gestures.
3. Parse the file into `t` and an N×12 array. Check sequence gaps, strictly increasing timestamps, finite values, and sample intervals. Split a recording at timestamp resets or long gaps instead of interpolating through them. Use `resample_linear` only on acceptable contiguous sections.
4. Plot and verify each channel before extracting features. Reserve entire recordings—and eventually entire wearers—for evaluation. Recalibrate when placement changes.

The production calibration code uses five-sample causal averaging, 24 signed-mean/RMS features, standardized 3-NN, and two consecutive matching labels before publishing a change. Our labs expose these ideas but are not a replacement for its serial handling or calibration workflow.

In [30]:
# OPTIONAL: deliberately inactive on Run All. Core labs need neither pyserial nor hardware.
RECORD_HARDWARE = False
if RECORD_HARDWARE:
    import time
    from pathlib import Path
    import serial
    port = '/dev/cu.usbserial-REPLACE_ME'  # Windows example: COM3
    output = Path('wrist_up_trial01_frames.txt')
    with output.open('x') as destination, serial.Serial(port,115200,timeout=.2) as connection:
        connection.reset_input_buffer()
        stop=time.monotonic()+6
        while time.monotonic()<stop:
            line=connection.readline().decode('utf-8',errors='replace').strip()
            if line.startswith('FRAME,'):
                destination.write(line+'\n')
    print('Saved',output)

## Further reading and course connections
The explanations and exercises here are original, with topic selection informed by these Berkeley courses:
- [EE 66: Introduction to Signals, Dynamics, and Information](https://www2.eecs.berkeley.edu/Courses/EE66/) — a linear-algebra foundation for signals and systems.
- [EECS 16A, Fall 2025](https://www-inst.eecs.berkeley.edu/~ee16a/fa25/) — signals as vectors, inner products, periodicity, and complex exponentials.
- [EE 120 course description](https://www2.eecs.berkeley.edu/Courses/EE120/) and [Fall 2026 course calendar](https://ee120-course-staff.github.io/) — LTI systems, convolution, impulse response, causality, and frequency response.

We borrow the conceptual progression, not the scope of a full semester. Continuous-time transforms, Laplace/z transforms, feedback design, and proofs of sampling reconstruction are beyond this workshop.

**Final reflection:** explain one GuaqBand failure using each of these lenses: acquisition/sampling, filtering/time response, feature representation, and evaluation design.

<a id="optional-geometry"></a>
## Optional extension · Waveforms as vectors
**Lecture companion: “Optional · Waveforms as vectors” (slide 121) → “Projection is a one-template model” (slide 122) → “Normalized similarity compares shape” (slide 123)**

Return to an abstract waveform and view its finite sequence as a vector. An inner product measures alignment with a reference waveform. This gives another way to understand the Fourier coefficients from Lab 4.

The least-squares coefficient $\alpha=(g^Tx)/(g^Tg)$ gives the multiple of nonzero g closest to x. Its residual, $x-\alpha g$, is orthogonal to g. Cosine similarity divides the dot product by both norms, retaining direction while discarding positive scale.

### Coding challenge · Compare waveforms
- `project(x, g)` returns `(alpha, residual)` for nonzero g.
- `cosine_scores(x, templates)` compares x with each template row. Return zero for comparisons involving a zero norm.

**Predict → run → explain:** compare a waveform, twice that waveform, and its negative. Which projection coefficients scale? Which similarities retain or reverse sign? Why would a comparison that ignores positive scale be useful in some tasks and inappropriate in others?

This extension does not gate any earlier lab.


In [31]:
def project(x, g):
    # TODO: compute the least-squares coefficient and orthogonal residual.
    raise NotImplementedError("project")

def cosine_scores(x, templates):
    # TODO: normalized inner products, with zero-norm comparisons defined as zero.
    raise NotImplementedError("cosine_scores")


In [32]:
# OPTIONAL: this cell does not gate any later lab.
try:
    basis_t = np.arange(100) / 100
    g = np.sin(2*np.pi*2*basis_t)
    q = np.cos(2*np.pi*2*basis_t)
    waveform = 2*g + .5*q
    alpha, residual = project(waveform, g)
    np.testing.assert_allclose(alpha, 2., atol=1e-12)
    assert abs(residual @ g) < 1e-10
    templates = np.vstack([g, q])
    np.testing.assert_allclose(cosine_scores(np.zeros(100), templates), [0, 0])
    np.testing.assert_allclose(cosine_scores(2*waveform, templates), cosine_scores(waveform, templates))
    print('Sine coefficient:', alpha, '| cosine coefficient:', project(waveform, q)[0])
    print('Dot products:', templates @ waveform)
    print('Cosine scores:', cosine_scores(waveform, templates))
except NotImplementedError as error:
    print('Optional waveform projection extension:', error)


Optional waveform projection extension: project


<details><summary>Worked explanation · compare after predicting</summary>

The example waveform is 2g + 0.5q, where the chosen sine and cosine are orthogonal over the record. Projecting onto g gives coefficient 2; projecting onto q gives 0.5. Doubling the waveform doubles those coefficients, while cosine similarities stay the same. Negating it reverses their signs. A zero vector has no direction, so the exercise explicitly defines its similarity as zero.

</details>
